In [2]:
import pandas as pd, numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_parquet("../out/ksa_sales_clean.parquet")

m = df[
    (df.classification == "سكني")
    & (~df.is_multi_property)
    & (df.area_m2.between(50, 5_000))
    & (df.price_sar.between(50_000, 20_000_000))
].copy()

print(f"{len(df):,} cleaned  ->  {len(m):,} in scope")

1,222,288 cleaned  ->  927,761 in scope


In [3]:
import pandas as pd, numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

cut = pd.Timestamp("2025-01-01")
tr, te = m[m.date_greg < cut], m[m.date_greg >= cut]
print(f"train {len(tr):,} | test {len(te):,}")

# --- fitted on TRAIN ONLY ---
tr_ppm   = tr.price_sar / tr.area_m2
counts   = tr.location.value_counts()
loc_med  = tr_ppm.groupby(tr.location).median()
loc_med  = loc_med[counts.reindex(loc_med.index).fillna(0) >= 10]
city_med = tr_ppm.groupby(tr.city).median()
glob_med = tr_ppm.median()

# --- predict ---
ppm_hat = te.location.map(loc_med)
print(f"fell back from location: {ppm_hat.isna().mean():.1%} of test rows")
ppm_hat = ppm_hat.fillna(te.city.map(city_med)).fillna(glob_med)
pred    = ppm_hat * te.area_m2

# --- score ---
y, ly, lp = te.price_sar, np.log1p(te.price_sar), np.log1p(pred)
print("\nBASELINE — median price/m² by location × area, test = 2025+")
print("  R²  (log) :", round(r2_score(ly, lp), 4))
print("  MAE (log) :", round(mean_absolute_error(ly, lp), 4))
print("  median abs % error:", f"{np.median(np.abs(pred - y) / y):.1%}")
print("  bias (median pred/actual):", round(float(np.median(pred / y)), 3))

train 773,674 | test 154,087
fell back from location: 6.0% of test rows

BASELINE — median price/m² by location × area, test = 2025+
  R²  (log) : 0.5363
  MAE (log) : 0.4893
  median abs % error: 31.1%
  bias (median pred/actual): 0.808
